# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AW-OMW/FLY-RANK-PROJECT/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Rule for Predicting Page Boom (Data-Driven) and its Reason Codes

### **Rule**:

A page is predicted to 'boom' next month if it meets *at least two* of the following criteria:
1.  `trend_last_3_months_views` is positive and significant (e.g., greater than 10%).
2.  `search_volume` is in the top 25% of all content items.
3.  `competition` is in the bottom 50% of all content items.

If a page does not meet at least two of these criteria, it is predicted to `not boom`.

### **Reason Codes**:

*   `BOOM_PREDICTED_STRONG_TREND_HIGH_SEARCH`: The page shows a strong positive view trend and high search volume.
*   `BOOM_PREDICTED_HIGH_SEARCH_LOW_COMPETITION`: The page has high search volume and relatively low competition.
*   `BOOM_PREDICTED_STRONG_TREND_LOW_COMPETITION`: The page exhibits a strong positive view trend and relatively low competition.
*   `NO_BOOM_PREDICTED_INSUFFICIENT_CRITERIA`: The page does not meet enough criteria for a 'boom' prediction.
*   `NO_BOOM_PREDICTED_LOW_TREND`: The page has a negative or stagnant view trend.
*   `NO_BOOM_PREDICTED_HIGH_COMPETITION`: The page faces high competition, hindering a potential boom.
*   `NO_BOOM_PREDICTED_LOW_SEARCH_VOLUME`: The page has a low search volume, indicating limited interest.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


In [1]:
display(df.columns)

NameError: name 'df' is not defined

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
#my rule is if a page has positive and significant trend in last 3 months , search volume is more than 25% and competition level is less than 50% the page will have high chance of booming
#if the page has atleast 2 of these criteria it will be marked as highly likely to boom

In [4]:
#reason codes are :
#BOOM_PREDICTED_STRONG_TREND_HIGH_SEARCH: The page shows a strong positive view trend and high search volume.
#BOOM_PREDICTED_HIGH_SEARCH_LOW_COMPETITION: The page has high search volume and relatively low competition.
#BOOM_PREDICTED_STRONG_TREND_LOW_COMPETITION: The page exhibits a strong positive view trend and relatively low competition.
#NO_BOOM_PREDICTED_INSUFFICIENT_CRITERIA: The page does not meet enough criteria for a 'boom' prediction.
#NO_BOOM_PREDICTED_LOW_TREND: The page has a negative or stagnant view trend.
#NO_BOOM_PREDICTED_HIGH_COMPETITION: The page faces high competition, hindering a potential boom.
#NO_BOOM_PREDICTED_LOW_SEARCH_VOLUME: The page has a low search volume, indicating limited interest.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Define and Apply the `calculate_boom_score` Function

This function will assess each page against the criteria to determine its 'boom' potential, returning a numerical score and associated reason codes. We'll dynamically calculate the `search_volume` and `competition` thresholds based on your DataFrame's data.

In [7]:
def calculate_boom_score(row, search_volume_threshold, competition_threshold):
    """Calculates a 'boom' score for a given row based on defined criteria.

    Args:
        row (pd.Series): A row from the DataFrame.
        search_volume_threshold (float): The 75th percentile of search_volume.
        competition_threshold (float): The 50th percentile of competition.

    Returns:
        tuple: A tuple containing the boom score (int) and a list of reason codes (list of str).
    """
    score = 0
    reason_codes = []

    # Condition 1: trend_pct is positive and significant (>10%)
    if pd.notna(row['trend_pct']) and row['trend_pct'] > 10:
        score += 1
        reason_codes.append('STRONG_TREND')

    # Condition 2: search_volume is in the top 25% of all content items
    if pd.notna(row['search_volume']) and row['search_volume'] > search_volume_threshold:
        score += 1
        reason_codes.append('HIGH_SEARCH_VOLUME')

    # Condition 3: competition is in the bottom 50% of all content items
    if pd.notna(row['competition']) and row['competition'] < competition_threshold:
        score += 1
        reason_codes.append('LOW_COMPETITION')

    return score, reason_codes

# Calculate dynamic thresholds from the DataFrame
search_volume_threshold = df['search_volume'].quantile(0.75)
competition_threshold = df['competition'].quantile(0.50)

print(f"Calculated Search Volume Threshold (75th percentile): {search_volume_threshold:.2f}")
print(f"Calculated Competition Threshold (50th percentile): {competition_threshold:.2f}")

Calculated Search Volume Threshold (75th percentile): 20.00
Calculated Competition Threshold (50th percentile): 0.00


In [8]:
# Apply the function to each row of the DataFrame
df[['boom_score', 'reason_codes']] = df.apply(
    lambda row: calculate_boom_score(row, search_volume_threshold, competition_threshold),
    axis=1,
    result_type='expand'
)

# Display the first few rows with the new scores and reason codes
display(df[['content_id', 'trend_pct', 'search_volume', 'competition', 'boom_score', 'reason_codes']].head())

,content_id,trend_pct,search_volume,competition,boom_score,reason_codes
0,content_304f48230142,-41.4,10.0,0.67,0,[]
1,content_a1fb4e703a9e,-57.7,90.0,0.01,1,[HIGH_SEARCH_VOLUME]
2,content_9aa793d4d895,-60.9,0.0,0.00,0,[]
3,content_331d6c4de07b,-13.8,10.0,0.00,0,[]
4,content_d99b7a2d90ca,-34.7,0.0,0.00,0,[]


In [9]:
import os

output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'baseline_action_score.csv')
df.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: work/outputs/baseline_action_score.csv


The `df` DataFrame, which now includes the `boom_score` and `reason_codes` for all 30,000 content items, has been saved to `work/outputs/baseline_action_score.csv`. This file contains the complete dataset with the calculated scores, ready for further analysis or ranking.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.